In [1]:
"""Complete training pipeline for defect detection models."""

import os
import sys
from pathlib import Path

from abbvisionsystem.training_pipeline.data_manager import organize_enhanced_dataset, prepare_enhanced_yolo_dataset, generate_synthetic_defects
from abbvisionsystem.training_pipeline.yolo_trainer import YOLODefectDetector, create_multi_object_test_images
from abbvisionsystem.training_pipeline.resnet_trainer import DefectClassificationModel

2025-09-02 22:12:27.971203: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-02 22:12:27.978332: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756825947.985960  285628 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756825947.988090  285628 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756825947.994025  285628 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
def run_complete_pipeline(
    source_data_dir: str,
    use_yolo: bool = True,
    use_classification: bool = True,
    train_yolo_epochs: int = 100,
    train_classification_epochs: int = 50,
    dataset_strategy: str = "mixed",
    balance_classes: bool = True,
    multi_object_scenes: int = 300
):
    """Run complete training pipeline for both YOLO and classification models with enhanced data handling."""
    
    print("🚀 Starting Enhanced Defect Detection Training Pipeline")
    print("=" * 60)
    print(f"📊 Dataset Strategy: {dataset_strategy}")
    print(f"⚖️  Class Balancing: {balance_classes}")
    print(f"🎬 Multi-object Scenes: {multi_object_scenes}")
    
    # Step 1: Organize enhanced dataset
    print("\n📁 Step 1: Organizing enhanced dataset...")
    classification_dataset = "training_data/enhanced_defect_detection_dataset"
    organize_enhanced_dataset(
        source_data_dir=source_data_dir,
        output_dir=classification_dataset,
        strategy=dataset_strategy,
        train_ratio=0.7,
        val_ratio=0.15,
        test_ratio=0.15,
        balance_classes=balance_classes
    )
    
    # Step 2: Prepare enhanced YOLO dataset
    print("\n🎯 Step 2: Preparing enhanced YOLO dataset...")
    yolo_dataset_yaml = prepare_enhanced_yolo_dataset(
        source_data_dir=source_data_dir,
        classification_dataset_dir=classification_dataset,
        output_dir="training_data/enhanced_yolo_dataset",
        multi_object_scenes=multi_object_scenes,
        use_legacy=False
    )
    
    # Step 3: Create multi-object test images
    print("\n🖼️ Step 3: Creating multi-object test images...")
    create_multi_object_test_images(
        f"{classification_dataset}/test",
        "multi_object_test",
        images_per_composition=50
    )
    
    results = {}
    
    # Step 4: Train YOLO model (FIXED - using correct detection models)
    if use_yolo:
        print("\n🤖 Step 4: Training Enhanced YOLO Detection model...")
        yolo_detector = YOLODefectDetector()
        
        # FIXED: Use proper detection models, not classification models
        detection_models = [
            "yolo11s.pt",     # Primary choice - YOLO11 small detection
            "yolov8s.pt",     # Fallback 1 - YOLOv8 small detection  
            "yolov8n.pt",     # Fallback 2 - YOLOv8 nano detection
            "yolo11n.pt"      # Fallback 3 - YOLO11 nano detection
        ]
        
        model_loaded = False
        loaded_model = None
        
        for model_file in detection_models:
            print(f"🔄 Trying to load {model_file}...")
            if yolo_detector.load_model(model_file):
                print(f"✅ Successfully loaded detection model: {model_file}")
                loaded_model = model_file
                model_loaded = True
                break
            else:
                print(f"⚠️  Failed to load {model_file}, trying next...")
        
        if not model_loaded:
            print("❌ Failed to load any YOLO detection model. Skipping YOLO training.")
            print("💡 Available models will be downloaded automatically during training.")
            # Try to proceed with default model
            try:
                yolo_detector.load_model("yolo11s.pt")
                model_loaded = True
                loaded_model = "yolo11s.pt"
                print("✅ Proceeding with auto-download of yolo11s.pt")
            except:
                results['yolo'] = None
        
        if model_loaded:
            try:
                print(f"📈 Training YOLO Detection Model:")
                print(f"   🎯 Model: {loaded_model}")
                print(f"   📊 Strategy: {dataset_strategy}")
                print(f"   🎬 Multi-object scenes: {multi_object_scenes}")
                print(f"   ⚖️  Class balancing: {balance_classes}")
                print(f"   🔍 Task: Object Detection (not classification)")
                
                best_yolo_weights = yolo_detector.train(
                    dataset_yaml=yolo_dataset_yaml,
                    epochs=train_yolo_epochs,
                    imgsz=640,
                    batch=16,
                    project='trained_models',
                    name='enhanced_yolo_defect_detector'
                )
                
                # Evaluate on your "both" dataset
                print("\n📊 Evaluating YOLO Detection model on real multi-object images...")
                yolo_results = evaluate_on_both_dataset(yolo_detector, f"{source_data_dir}/both")
                results['yolo'] = yolo_results
                
                print(f"🎯 YOLO Detection Training Completed:")
                print(f"   📄 Best weights: {best_yolo_weights}")
                print(f"   🎯 Model type: Object Detection")
                print(f"   📊 Trained on enhanced dataset with {multi_object_scenes} multi-object scenes")
                
            except Exception as e:
                print(f"❌ YOLO Detection training failed: {e}")
                print("💡 Possible solutions:")
                print("   - Ensure you're using detection model (.pt), not classification (-cls.pt)")
                print("   - Try reducing batch size (batch=8 or batch=4)")
                print("   - Reduce image size (imgsz=320)")
                print("   - Ensure sufficient disk space")
                print("   - Check GPU memory (training will use CPU as fallback)")
                print(f"   - Verify dataset format in {yolo_dataset_yaml}")
                results['yolo'] = None
    
    # Step 5: Train classification model (for comparison)
    if use_classification:
        print("\n🧠 Step 5: Training Enhanced ResNet50V2 classification model...")
        classifier = DefectClassificationModel()
        classifier.build_model()
        
        try:
            # Prepare data with enhanced dataset
            train_gen, val_gen = classifier.prepare_data_generators(
                f"{classification_dataset}/train",
                f"{classification_dataset}/validation"
            )
            
            print(f"📊 Classification Training data prepared:")
            print(f"   Training samples: {train_gen.samples}")
            print(f"   Validation samples: {val_gen.samples}")
            
            # Train
            classifier.train(
                train_gen, val_gen,
                epochs=train_classification_epochs,
                model_name="enhanced_resnet_defect_classifier"
            )
            
            # Evaluate
            test_gen = classifier.prepare_data_generators(
                f"{classification_dataset}/test",
                f"{classification_dataset}/test"
            )[1]  # Use validation generator (no augmentation)
            
            classification_results = classifier.evaluate(test_gen)
            results['classification'] = classification_results
            
            # Save model
            classifier.save_model("enhanced_resnet_defect_classifier")
            
            print(f"🧠 Enhanced Classification Results:")
            print(f"  Test Accuracy: {classification_results['test_accuracy']:.4f}")
            print(f"  Test Precision: {classification_results['test_precision']:.4f}")
            print(f"  Test Recall: {classification_results['test_recall']:.4f}")
            
        except Exception as e:
            print(f"❌ Classification training failed: {e}")
            results['classification'] = None
    
    # Step 6: Enhanced model comparison
    print("\n📈 Step 6: Enhanced Model Comparison Summary")
    print("=" * 50)
    
    if results.get('yolo') and results.get('classification'):
        print("🏆 Model Performance Comparison (Enhanced Dataset):")
        print("   YOLO = Object Detection | ResNet = Classification")
        print(f"{'Metric':<15} {'YOLO (Detect)':<15} {'ResNet (Class)':<15} {'Best':<8}")
        print("-" * 53)
        
        # Compare accuracy
        yolo_acc = results['yolo']['accuracy']
        class_acc = results['classification']['test_accuracy']
        best_acc = "YOLO" if yolo_acc > class_acc else "ResNet"
        print(f"{'Accuracy':<15} {yolo_acc:<15.4f} {class_acc:<15.4f} {best_acc:<8}")
        
        # Compare precision
        yolo_prec = results['yolo']['precision']
        class_prec = results['classification']['test_precision']
        best_prec = "YOLO" if yolo_prec > class_prec else "ResNet"
        print(f"{'Precision':<15} {yolo_prec:<15.4f} {class_prec:<15.4f} {best_prec:<8}")
        
        # Compare recall
        yolo_recall = results['yolo']['recall']
        class_recall = results['classification']['test_recall']
        best_recall = "YOLO" if yolo_recall > class_recall else "ResNet"
        print(f"{'Recall':<15} {yolo_recall:<15.4f} {class_recall:<15.4f} {best_recall:<8}")
        
        # Calculate and compare F1 scores
        yolo_f1 = results['yolo']['f1_score']
        precision = results['classification']['test_precision']
        recall = results['classification']['test_recall']
        class_f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        best_f1 = "YOLO" if yolo_f1 > class_f1 else "ResNet"
        print(f"{'F1 Score':<15} {yolo_f1:<15.4f} {class_f1:<15.4f} {best_f1:<8}")
        
        # Additional YOLO-specific metrics
        print(f"\n🎯 YOLO Object Detection Specific Metrics:")
        print(f"  Detection Rate: {results['yolo']['detection_rate']:.4f}")
        print(f"  Avg Detections/Image: {results['yolo']['avg_detections_per_image']:.2f}")
        print(f"  Total Detections: {results['yolo']['total_detections']}")
        
    elif results.get('yolo'):
        print("🎯 YOLO Object Detection Results (Enhanced Dataset):")
        yolo_results = results['yolo']
        print(f"  Accuracy: {yolo_results['accuracy']:.4f}")
        print(f"  Precision: {yolo_results['precision']:.4f}")
        print(f"  Recall: {yolo_results['recall']:.4f}")
        print(f"  F1 Score: {yolo_results['f1_score']:.4f}")
        print(f"  Detection Rate: {yolo_results['detection_rate']:.4f}")
        
    elif results.get('classification'):
        print("🧠 Classification Model Results (Enhanced Dataset):")
        class_results = results['classification']
        print(f"  Accuracy: {class_results['test_accuracy']:.4f}")
        print(f"  Precision: {class_results['test_precision']:.4f}")
        print(f"  Recall: {class_results['test_recall']:.4f}")
    
    print("\n✅ Enhanced Pipeline completed successfully!")
    print("\n🎯 ENHANCED RECOMMENDATION FOR YOUR USE CASE:")
    print("With your rich dataset and multi-object detection requirements:")
    print("  • 🔍 YOLO (Object Detection) - Locates AND classifies multiple defects")
    print("  • 🧠 ResNet (Classification) - Single image-level classification")
    print("  • 🌄 Enhanced training on real backgrounds from your colorless datasets")
    print("  • 🔄 Handles deformed defects from defect_colorless_deform")
    print("  • 📝 Processes images without text from defect_colorless_nowords")
    print("  • 🎭 Creates realistic multi-object scenes")
    print("  • ⚖️  Balanced training data across all categories")
    print("  • 🎯 Optimized for real-world production scenarios")
    
    return results

In [3]:
def evaluate_on_both_dataset(yolo_detector, both_images_dir):
    """Evaluate on your 'both' dataset with multiple objects."""
    if not os.path.exists(both_images_dir):
        print(f"⚠️  'both' dataset directory not found: {both_images_dir}")
        # Return default metrics structure to avoid comparison errors
        return {
            "total_images": 0,
            "images_with_detections": 0,
            "total_detections": 0,
            "avg_detections_per_image": 0.0,
            "confidence_scores": [],
            "detection_rate": 0.0,
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "f1_score": 0.0
        }
    
    results = {
        "total_images": 0,
        "images_with_detections": 0,
        "total_detections": 0,
        "avg_detections_per_image": 0.0,
        "confidence_scores": [],
        "detection_rate": 0.0,
        "accuracy": 0.0,
        "precision": 0.0,
        "recall": 0.0,
        "f1_score": 0.0
    }
    
    image_files = [f for f in os.listdir(both_images_dir) 
                   if f.endswith(('.jpg', '.jpeg', '.png', '.JPG'))]
    
    # Metrics tracking
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    true_negatives = 0
    
    for img_file in image_files:
        img_path = os.path.join(both_images_dir, img_file)
        
        try:
            detections = yolo_detector.predict(img_path, conf_threshold=0.25)
            
            results["total_images"] += 1
            num_detections = len(detections["boxes"])
            defect_detections = sum(1 for cls in detections["classes"] if cls == 1)
            
            if num_detections > 0:
                results["images_with_detections"] += 1
                results["total_detections"] += num_detections
                results["confidence_scores"].extend(detections["scores"])
            
            # For evaluation, assume images with "defect" in filename are defective
            # You may need to adjust this logic based on your actual labeling
            is_defective_image = "defect" in img_file.lower() or "bad" in img_file.lower()
            
            if is_defective_image and defect_detections > 0:
                true_positives += 1
            elif is_defective_image and defect_detections == 0:
                false_negatives += 1
            elif not is_defective_image and defect_detections > 0:
                false_positives += 1
            elif not is_defective_image and defect_detections == 0:
                true_negatives += 1
                
        except Exception as e:
            print(f"❌ Error processing {img_file}: {str(e)}")
            continue
    
    # Calculate metrics
    if results["total_images"] > 0:
        results["avg_detections_per_image"] = results["total_detections"] / results["total_images"]
        results["detection_rate"] = results["images_with_detections"] / results["total_images"]
        
        # Calculate classification metrics
        total_predictions = true_positives + false_positives + false_negatives + true_negatives
        if total_predictions > 0:
            results["accuracy"] = (true_positives + true_negatives) / total_predictions
        
        if true_positives + false_positives > 0:
            results["precision"] = true_positives / (true_positives + false_positives)
        
        if true_positives + false_negatives > 0:
            results["recall"] = true_positives / (true_positives + false_negatives)
        
        if results["precision"] + results["recall"] > 0:
            results["f1_score"] = 2 * (results["precision"] * results["recall"]) / (results["precision"] + results["recall"])
    
    print(f"📊 YOLO Evaluation Results on 'both' dataset:")
    print(f"   Total images: {results['total_images']}")
    print(f"   Images with detections: {results['images_with_detections']}")
    print(f"   Total detections: {results['total_detections']}")
    print(f"   Detection rate: {results['detection_rate']:.4f}")
    print(f"   Accuracy: {results['accuracy']:.4f}")
    print(f"   Precision: {results['precision']:.4f}")
    print(f"   Recall: {results['recall']:.4f}")
    print(f"   F1 Score: {results['f1_score']:.4f}")
    
    return results

In [4]:
def test_pipeline_setup():
    """Test if all components are properly set up for enhanced functionality."""
    print("🔍 Testing enhanced pipeline setup...")
    
    try:
        from abbvisionsystem.training_pipeline.data_manager import organize_enhanced_dataset, prepare_enhanced_yolo_dataset
        print("✅ Enhanced data_manager imports successful")
    except ImportError as e:
        print(f"❌ Enhanced data_manager import failed: {e}")
        return False
    
    try:
        from abbvisionsystem.training_pipeline.yolo_trainer import YOLODefectDetector
        print("✅ yolo_trainer import successful")
    except ImportError as e:
        print(f"❌ yolo_trainer import failed: {e}")
        return False
    
    try:
        from abbvisionsystem.training_pipeline.resnet_trainer import DefectClassificationModel
        print("✅ resnet_trainer import successful")
    except ImportError as e:
        print(f"❌ resnet_trainer import failed: {e}")
        return False
    
    # Check if ultralytics is available for YOLO
    try:
        from ultralytics import YOLO
        print("✅ ultralytics available")
        
        # Test model loading - FIXED: Test detection models, not classification
        yolo_detector = YOLODefectDetector()
        detection_models = [
            "yolo11s.pt",
            # "yolov8s.pt",
            # "yolov8n.pt",
            # "yolo11n.pt"
        ]
        
        model_available = False
        
        for model in detection_models:
            try:
                print(f"   🔄 Testing {model}...")
                if yolo_detector.load_model(model):
                    print(f"✅ YOLO detection model {model} loadable")
                    model_available = True
                    break
                else:
                    print(f"⚠️  {model} not available locally")
            except Exception as e:
                print(f"⚠️  {model} failed to load: {str(e)}")
                continue
        
        if not model_available:
            print("⚠️  No YOLO detection models could be loaded locally.")
            print("   💡 Models will be downloaded automatically during training.")
            print("   🎯 This is normal for first-time setup.")
        
    except ImportError:
        print("❌ ultralytics not installed. Install with: pip install ultralytics")
        return False
    
    # Check if tensorflow is available
    try:
        import tensorflow as tf
        print(f"✅ tensorflow {tf.__version__} available")
    except ImportError:
        print("❌ tensorflow not installed")
        return False
    
    # Test enhanced data manager functionality
    try:
        from abbvisionsystem.training_pipeline.data_manager import EnhancedDataManager
        print("✅ EnhancedDataManager class available")
    except ImportError as e:
        print(f"❌ EnhancedDataManager import failed: {e}")
        return False
    
    print("✅ Enhanced pipeline setup test completed successfully!")
    print("🎯 Ready for enhanced training with rich dataset support!")
    print("📋 Note: Using DETECTION models (.pt), not classification (-cls.pt)")
    return True


In [5]:
if __name__ == "__main__":
    # First test the setup
    if not test_pipeline_setup():
        print("❌ Setup test failed. Please fix the issues above.")
        exit(1)
    
    # Run the enhanced pipeline
    source_dir = "data/choco-pie"  # Update this path
    
    if not os.path.exists(source_dir):
        print(f"Source directory {source_dir} not found!")
        print("Please update the source_dir variable to point to your data.")
        print("Expected enhanced structure:")
        print("data/choco-pie/")
        print("├── good/                    # Cropped good images")
        print("├── defect/                  # Cropped defect images")
        print("├── good_colorless/          # Good images with backgrounds")
        print("├── defect_colorless/        # Defect images with backgrounds")
        print("├── defect_colorless_deform/ # Deformed defects with backgrounds")
        print("├── defect_colorless_nowords/# Defects without text")
        print("└── both/                    # Mixed test images")
    else:
        # Enhanced data structure analysis
        print("🔍 Analyzing enhanced data structure...")
        
        # Check for enhanced directories
        enhanced_dirs = [
            "good", "defect", "good_colorless", "defect_colorless", 
            "defect_colorless_deform", "defect_colorless_nowords", "both"
        ]
        
        available_dirs = []
        for dir_name in enhanced_dirs:
            dir_path = os.path.join(source_dir, dir_name)
            if os.path.exists(dir_path):
                available_dirs.append(dir_name)
        
        print(f"📊 Available data directories: {', '.join(available_dirs)}")
        
        # Determine best strategy based on available data
        has_cropped = any(d in available_dirs for d in ["good", "defect"])
        has_background = any(d in available_dirs for d in ["good_colorless", "defect_colorless", 
                                                          "defect_colorless_deform", "defect_colorless_nowords"])
        
        if has_cropped and has_background:
            strategy = "mixed"
            print("🎯 Using MIXED strategy - leveraging all available data")
        elif has_background:
            strategy = "background_only" 
            print("🌄 Using BACKGROUND_ONLY strategy - using images with backgrounds")
        elif has_cropped:
            strategy = "cropped_only"
            print("📸 Using CROPPED_ONLY strategy - using cropped images")
        else:
            strategy = "legacy"
            print("⚠️  Falling back to LEGACY strategy - basic good/defect structure")
        
        # Run enhanced pipeline with optimal settings
        results = run_complete_pipeline(
            source_data_dir=source_dir,
            use_yolo=True,
            use_classification=True,
            train_yolo_epochs=100,  # Increased for better performance with rich data
            train_classification_epochs=50,
            dataset_strategy=strategy,  # Automatically determined strategy
            balance_classes=True,      # Balance classes for better training
            multi_object_scenes=400   # More scenes for better generalization
        )
        
        # Additional evaluation on enhanced test set
        if results.get('yolo') and os.path.exists(os.path.join(source_dir, "both")):
            print("\n🧪 Additional Enhanced Evaluation:")
            print("Testing on 'both' dataset with real multi-object scenarios...")


🔍 Testing enhanced pipeline setup...
✅ Enhanced data_manager imports successful
✅ yolo_trainer import successful
✅ resnet_trainer import successful
✅ ultralytics available
   🔄 Testing yolo11s.pt...
Model loaded from yolo11s.pt
✅ YOLO detection model yolo11s.pt loadable
✅ tensorflow 2.19.0 available
✅ EnhancedDataManager class available
✅ Enhanced pipeline setup test completed successfully!
🎯 Ready for enhanced training with rich dataset support!
📋 Note: Using DETECTION models (.pt), not classification (-cls.pt)
🔍 Analyzing enhanced data structure...
📊 Available data directories: good, defect, good_colorless, defect_colorless, defect_colorless_deform, defect_colorless_nowords, both
🎯 Using MIXED strategy - leveraging all available data
🚀 Starting Enhanced Defect Detection Training Pipeline
📊 Dataset Strategy: mixed
⚖️  Class Balancing: True
🎬 Multi-object Scenes: 400

📁 Step 1: Organizing enhanced dataset...
🗂️  Enhanced Data Structure Analysis

📋 Cropped Images (no background):
     G

train: Scanning /mnt/mainhold/Downloads-Main/abb-capstone/training_data/enhanced_yolo_dataset/labels/train... 528 images, 107 backgrounds, 0 corrupt: 100%|██████████| 528/528 [00:00<00:00, 9168.14it/s]

train: New cache created: /mnt/mainhold/Downloads-Main/abb-capstone/training_data/enhanced_yolo_dataset/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 8850.4±2379.6 MB/s, size: 283.0 KB)


/home/dealoux/.cache/pypoetry/virtualenvs/abbvisionsystem-7xechF3d-py3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning /mnt/mainhold/Downloads-Main/abb-capstone/training_data/enhanced_yolo_dataset/labels/val... 132 images, 23 backgrounds, 0 corrupt: 100%|██████████| 132/132 [00:00<00:00, 9329.00it/s]

...

      3/100         0G   0.005607          1      1.428         39        640: 100%|██████████| 33/33 [01:46<00:00,  3.23s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:09<00:00,  1.95s/it]

                   all        132        115      0.838      0.947      0.949      0.809



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      4/100         0G   0.005403     0.9523      1.417         46        640:  88%|████████▊ | 29/33 [01:33<00:12,  3.23s/it]


KeyboardInterrupt: 